# Chapitre 10 - Cours 4 : au-delà de MapReduce, moteurs dataflow et langages de requête

MapReduce matérialise tout, trie entre chaque étape et attend la dernière tâche avant de
continuer. Les **moteurs dataflow** (Spark, Flink, Tez) traitent un workflow entier
comme un seul job, enchaînent les opérateurs sans tout écrire sur disque, et retrouvent
la tolérance aux pannes par **recalcul**. Par-dessus, les **langages de haut niveau**
(SQL) laissent un optimiseur choisir comment exécuter. Ce cours couvre
la section *Beyond MapReduce* (*Materialization of Intermediate State*, *Graphs and Iterative Processing*, *High-Level APIs and Languages*).

**Plan.**

1. Matérialiser l'état intermédiaire, ou le faire circuler.
2. Un moteur dataflow miniature : DAG d'opérateurs et fusion.
3. Tolérance aux pannes : lignage, recalcul et déterminisme.
4. Graphes et traitement itératif : le modèle Pregel.
5. Langages de haut niveau et optimiseurs.


In [1]:
import json
import math
import os
import sqlite3
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import batch as B
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
lines = B.generate_access_log(120_000, n_urls=1500, seed=3)
shards = [lines[i::6] for i in range(6)]